# Baseline 结果审计与绘图（2026-09-12）
从仓库根目录运行。图像由 `yumeow_plot.get_fig` 创建，分别导出 PDF、PNG、SVG。只在同一来源、同一任务/指标/骨干下画性能比较；跨论文冲突单独展示。

In [ ]:
import json
import re
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from yumeow_plot import get_fig

root = Path.cwd()
source = root / 'baseline/results.jsonl'
figdir = root / 'baseline/figs'
figdir.mkdir(parents=True, exist_ok=True)
rows = [json.loads(line) for line in source.open(encoding='utf-8') if line.strip()]
required = {'algorithm', 'data_split', 'metric', 'model', 'budget', 'result'}
assert all(required <= row.keys() for row in rows)
assert all(isinstance(row['result'], (int, float)) for row in rows)
key = lambda r: (r['algorithm'], r['data_split'], r['metric'], r['model'])
assert len({key(r) for r in rows}) == len(rows)
idx = {key(r): r for r in rows}
papers = Counter(r['algorithm'] for r in rows)
bench = re.compile(r'LLM[- ]?SRBench|LSR[- ](?:Synth|Transform)|LLMSR', re.I)
bench_papers = sorted({r['algorithm'] for r in rows if bench.search(r['data_split'])})
metrics = Counter(r['metric'] for r in rows)
budget_keys = Counter(k for r in rows for k in r['budget'])
print('总记录 / 文章 / data_split / metric:', len(rows), len(papers), len({r['data_split'] for r in rows}), len(metrics))
print('LLM-SRBench 文章:', len(bench_papers), ', '.join(bench_papers))
print('每篇条数:', ', '.join(f'{p}={n}' for p, n in sorted(papers.items())))
print('最常用指标:', metrics.most_common(20))
print('空 budget:', sum(not r['budget'] for r in rows), '；预算字段前十:', budget_keys.most_common(10))
print('PDF 数量:', len(list((root / 'baseline/paper').glob('*.pdf'))))
by_paper = defaultdict(set)
for row in rows:
    by_paper[row['algorithm']].add(row['data_split'].split(' @ ')[-1])
print('各文章数据源/实验后缀（含消融与重复实验）:')
for paper, suffixes in sorted(by_paper.items()):
    print(paper, ':', '; '.join(sorted(suffixes)))
# 跨论文筛查：仅以 @ 前的名义任务、原始指标名、原始模型名作严格字符串匹配。
# 这是待复核候选，不代表数据点、测试网格或预算已经一致。
nominal = defaultdict(lambda: defaultdict(set))
for row in rows:
    task = row['data_split'].split(' @ ')[0]
    nominal[(task, row['metric'], row['model'])][row['algorithm']].add(row['result'])
cross_paper = {k: v for k, v in nominal.items() if len(v) >= 2}
conflicts = {k: v for k, v in cross_paper.items() if len(set().union(*v.values())) > 1}
print('名义跨论文同方法/指标/任务组:', len(cross_paper), '；值不一致组:', len(conflicts))
for k, v in sorted(conflicts.items()):
    print('待复核:', k, {p: sorted(values) for p, values in v.items()})

def record(paper, split, metric, model):
    return idx[(paper, split, metric, model)]

def save_three(fig, stem):
    for ext in ('pdf', 'png', 'svg'):
        fig.savefig(figdir / f'{stem}.{ext}', dpi=300, bbox_inches='tight')
    plt.close(fig)


In [ ]:
# 同一 A-SR 表的 LSR-Transform 聚合：比较两个骨干下的 Acc@0.01。
split = 'LSR-Transform train/validation aggregate @ LLM-SRBench'
methods = ['LLMDirect', 'LLM-SR', 'LASR', 'SGA', 'DE', 'A-SR-Static', 'A-SR']
backbones = ['Llama3.1-8B-Instruct', 'Qwen3-4B-Instruct-2507']
acc = np.array([[record('A-SR', split, 'Acc@0.01', f'{m} ({b})')['result'] for m in methods] for b in backbones])
nmse = np.array([[record('A-SR', split, 'NMSE', f'{m} ({b})')['result'] for m in methods] for b in backbones])
print('A-SR 表，Acc@0.01:', dict(zip(backbones, acc.tolist())))
print('A-SR 表，NMSE:', dict(zip(backbones, nmse.tolist())))
fig, axes = get_fig(RN=1, CN=1, AW=14, AH=8, LM=5, RM=5, TM=5, BM=5, HS=5, VS=5, fontsize=14, dpi=300, lw=1)
ax = axes[0]
x = np.arange(len(methods))
ax.bar(x - 0.2, acc[0], 0.39, label='Llama 3.1 8B', color='#3274A1')
ax.bar(x + 0.2, acc[1], 0.39, label='Qwen3 4B', color='#E1812C')
ax.set_xticks(x, methods, rotation=35, ha='right', fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Acc@0.01 (higher is better)', fontsize=14)
ax.set_title('LSR-Transform, same source table', fontsize=14)
ax.legend(fontsize=10, frameon=False)
ax.grid(axis='y', alpha=0.25)
save_three(fig, 'transform_acc_backbone')


In [ ]:
# 精度双目标：仅比较 A-SR 同一表、同一骨干的 NMSE 与 Acc@0.01。
fig, axes = get_fig(RN=1, CN=2, AW=8, AH=7, LM=5, RM=5, TM=5, BM=5, HS=5, VS=5, fontsize=14, dpi=300, lw=1)
colors = plt.get_cmap('tab10').colors
for j, ax in enumerate(axes):
    for i, method in enumerate(methods):
        ax.scatter(nmse[j, i], acc[j, i], s=75, c=[colors[i]], label=method, zorder=3)
    frontier = [i for i in range(len(methods)) if not any((nmse[j, k] <= nmse[j, i] and acc[j, k] >= acc[j, i] and (nmse[j, k] < nmse[j, i] or acc[j, k] > acc[j, i])) for k in range(len(methods)))]
    frontier.sort(key=lambda i: nmse[j, i])
    ax.plot(nmse[j, frontier], acc[j, frontier], '--', color='black', lw=1, alpha=0.65)
    ax.set_title(['Llama 3.1 8B', 'Qwen3 4B'][j], fontsize=14)
    ax.set_xlabel('NMSE (lower is better)', fontsize=14)
    ax.set_ylabel('Acc@0.01 (higher is better)', fontsize=14)
    ax.grid(alpha=0.2)
axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=10, frameon=False)
save_three(fig, 'transform_accuracy_pareto')
print('精度 Pareto，Llama / Qwen:', [[methods[i] for i in range(len(methods)) if not any((nmse[j,k] <= nmse[j,i] and acc[j,k] >= acc[j,i] and (nmse[j,k] < nmse[j,i] or acc[j,k] > acc[j,i])) for k in range(len(methods)))] for j in range(2)])


In [ ]:
# 同一 LLM-PySR 扩展表：AI-Feynman 74 个方程上的 token-精度 Pareto。
pysr_split = '74 equations @ AI-Feynman'
token_models = ['LLM-PySR (search controller; GPT-5.1)', 'Equation author (GPT-5.1)', 'Candidate decider (GPT-5.1)', 'LLM-SR (full-LLM; GPT-5.1)', 'DrSR (full-LLM; GPT-5.1)']
short_names = ['LLM-PySR', 'Equation author', 'Candidate decider', 'LLM-SR', 'DrSR']
token_rows = [record('LLM-PySR', pysr_split, 'Mean ACC@0.1', m) for m in token_models]
tokens = np.array([r['budget']['token_usage'] for r in token_rows])
scores = np.array([r['result'] for r in token_rows])
fig, axes = get_fig(RN=1, CN=1, AW=11, AH=8, LM=5, RM=5, TM=5, BM=5, HS=5, VS=5, fontsize=14, dpi=300, lw=1)
ax = axes[0]
for i, label in enumerate(short_names):
    ax.scatter(tokens[i], scores[i], s=85, color=colors[i], label=label, zorder=3)
frontier = [i for i in range(len(tokens)) if not any((tokens[k] <= tokens[i] and scores[k] >= scores[i] and (tokens[k] < tokens[i] or scores[k] > scores[i])) for k in range(len(tokens)))]
frontier.sort(key=lambda i: tokens[i])
ax.plot(tokens[frontier], scores[frontier], '--', color='black', lw=1)
ax.set_xscale('log')
ax.set_xlim(5e4, 5e6)
ax.set_ylim(0.55, 1.02)
ax.set_xlabel('Mean LLM tokens per equation (lower is better)', fontsize=14)
ax.set_ylabel('Mean ACC@0.1 (higher is better)', fontsize=14)
ax.grid(alpha=0.2)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=10, frameon=False)
save_three(fig, 'aifeynman_token_pareto')
print('token-精度 Pareto:', [short_names[i] for i in frontier])


In [ ]:
# LLM-SR 原文 vs CoEvo 同名 GPT-3.5 方法：逐任务/切分的绝对差异。
tasks = ['Oscillation 1', 'Oscillation 2', 'E. coli growth', 'Stress-Strain']
pairs = []
for task in tasks:
    for split_kind in ['ID', 'OOD']:
        original = record('LLM-SR', f'{task} {split_kind} @ scientific benchmark', 'NMSE', 'LLM-SR (GPT-3.5)')
        coevo = record('CoEvo', f'{task} {split_kind} @ LLM-SR scientific benchmark', 'NMSE', 'LLM-SR (GPT-3.5-turbo)')
        pairs.append((task, split_kind, original['result'], coevo['result']))
print('跨论文同名方法原文/CoEvo NMSE 和倍数:')
for task, kind, a, b in pairs:
    print(task, kind, a, b, f'{b/a:.3g}x')
pysr_original = record('LLM-SR', 'Stress-Strain ID @ scientific benchmark', 'NMSE', 'PySR')['result']
pysr_pg = record('PG-SR', 'Stress-Strain ID @ LLM-SRBench', 'NMSE', 'PySR')['result']
llmsr_pg = record('PG-SR', 'Stress-Strain ID @ LLM-SRBench', 'NMSE', 'LLM-SR (GPT-4o-mini)')['result']
print('Stress-Strain ID / PySR：原文、PG-SR、倍数:', pysr_original, pysr_pg, pysr_original/pysr_pg)
print('名义排序：原文 LLM-SR GPT-3.5 < PySR?', record('LLM-SR', 'Stress-Strain ID @ scientific benchmark', 'NMSE', 'LLM-SR (GPT-3.5)')['result'] < pysr_original)
print('名义排序：PG-SR 中 LLM-SR GPT-4o-mini < PySR?', llmsr_pg < pysr_pg)
fig, axes = get_fig(RN=1, CN=1, AW=9, AH=9, LM=5, RM=5, TM=5, BM=5, HS=5, VS=5, fontsize=14, dpi=300, lw=1)
ax = axes[0]
for i, (task, kind, a, b) in enumerate(pairs):
    ax.scatter(a, b, c=['#3274A1' if kind == 'ID' else '#E1812C'], s=75, marker='o' if kind == 'ID' else 's', label=kind if i < 2 else None, zorder=3)
ax.plot([1e-10, 1], [1e-10, 1], '--', color='black', lw=1, label='Agreement')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(1e-10, 1)
ax.set_ylim(1e-10, 1)
ax.set_xlabel('LLM-SR paper NMSE', fontsize=14)
ax.set_ylabel('CoEvo paper NMSE', fontsize=14)
ax.grid(alpha=0.2)
ax.legend(fontsize=10, frameon=False)
save_three(fig, 'llmsr_coevo_discrepancy')
